# NB_LIBRO_00 · Empaquetar resultados para el manuscrito (versión 2)

Reúne en un solo archivo zip todo lo que el manuscrito necesita y lo descarga a su computadora:

- las figuras de `AGROVISION_LIBRO/02_FIGURAS` de los capítulos indicados en `CAPITULOS`;
- los valores y tablas de `AGROVISION_LIBRO/03_VALORES`;
- las figuras que ya existían en los proyectos de palma y pollos (lista `EXISTENTES`), si corresponden a esos capítulos.

Novedad de esta versión: el zip incluye solo los capítulos de la lista `CAPITULOS`, para que no crezca con cada capítulo nuevo. Con `CAPITULOS = None` se incluyen todos.

**Uso:** ejecute primero el notebook de figuras del capítulo que corresponda, luego este (*Entorno de ejecución → Ejecutar todo*) y suba el zip descargado al chat.

In [ ]:
import os, zipfile, datetime
from pathlib import Path
from PIL import Image

try:
    from google.colab import drive, files
    drive.mount('/content/drive')
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

BASE = Path(os.environ.get('AGROVISION_BASE', '/content/drive/MyDrive'))
DIR_LIBRO = BASE / 'AGROVISION_LIBRO'
DIR_PAQ = DIR_LIBRO / '04_PAQUETES'
DIR_PAQ.mkdir(parents=True, exist_ok=True)

# Capítulos que entran en el zip. Ejemplos: ['cap03', 'cap04'] · ['cap01', 'cap02'] · None (todos)
CAPITULOS = ['cap03', 'cap04']

# Figuras que ya existían en los proyectos: nombre dentro del zip -> archivo original.
EXISTENTES = {
    'cap01/Fig_1_3_mosaico_12_escenas.jpg':
        BASE / 'AgroUTMACH_PIX' / '05_RESULTADOS' / 'NB03_VIDEO' / 'NB03_mosaico_12_escenas.png',
}
LADO_MAX = 2400          # píxeles del lado mayor de las figuras existentes dentro del zip

assert DIR_LIBRO.exists(), f'No existe {DIR_LIBRO}. Ejecute antes el notebook de figuras.'

In [ ]:
sello = datetime.datetime.now().strftime('%Y%m%d_%H%M')
ruta_zip = DIR_PAQ / f'AgroVision_resultados_{sello}.zip'
incluidos, faltantes = [], []

with zipfile.ZipFile(ruta_zip, 'w', zipfile.ZIP_DEFLATED) as z:
    # 1) Figuras y valores generados por los notebooks del libro
    for carpeta in ('02_FIGURAS', '03_VALORES'):
        for f in sorted((DIR_LIBRO / carpeta).rglob('*')):
            if carpeta == '02_FIGURAS' and CAPITULOS is not None and f.parent.name not in CAPITULOS:
                continue
            if f.is_file():
                nombre = str(f.relative_to(DIR_LIBRO)).replace(os.sep, '/')
                z.write(f, nombre)
                incluidos.append((nombre, f.stat().st_size))
    # 2) Figuras existentes, reducidas y en JPEG
    for nombre, origen in EXISTENTES.items():
        if CAPITULOS is not None and nombre.split('/')[0] not in CAPITULOS:
            continue
        if not origen.exists():
            faltantes.append(str(origen))
            continue
        tmp = Path('/tmp') / Path(nombre).name
        with Image.open(origen) as im:
            original = im.size
            im = im.convert('RGB')
            im.thumbnail((LADO_MAX, LADO_MAX), Image.LANCZOS)
            im.save(tmp, 'JPEG', quality=92, dpi=(300, 300))
        destino = '02_FIGURAS/' + nombre
        z.write(tmp, destino)
        incluidos.append((destino + f'  (original {original[0]} × {original[1]})', tmp.stat().st_size))
    # 3) Listado del contenido
    lineas = [f'AgroVisión · paquete de resultados · {sello}', '']
    lineas += [f'{t / 1e6:8.2f} MB  {n}' for n, t in incluidos]
    lineas += ['', 'NO ENCONTRADOS:'] + (faltantes or ['(ninguno)'])
    z.writestr('CONTENIDO.txt', '\n'.join(lineas))

print('\n'.join(lineas))
print(f'\nZip: {ruta_zip}')
print(f'Tamaño: {ruta_zip.stat().st_size / 1e6:.1f} MB · {len(incluidos)} archivos')
n_fig = sum(1 for n, _ in incluidos if n.startswith('02_FIGURAS/'))
print(f'Figuras incluidas: {n_fig} · capítulos: {CAPITULOS or "todos"}')
if n_fig == 0:
    print('AVISO: no hay figuras de esos capítulos. ¿Ejecutó antes el notebook de figuras?')

In [ ]:
# Descarga el zip a su computadora. También queda guardado en AGROVISION_LIBRO/04_PAQUETES.
if EN_COLAB:
    files.download(str(ruta_zip))
else:
    print('Fuera de Colab: el zip está en', ruta_zip)